In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import hddm
import numpy as np
from pandas import Series
import sys
import os
import seaborn as sns

from kabuki.analyze import check_geweke
from patsy import dmatrix

print(hddm.__version__)
os.getcwd()

0.8.0


/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/IPython/parallel.py:13: ShimWarning: The `IPython.parallel` package has been deprecated since IPython 4.0. You should import from ipyparallel instead.
  "You should import from ipyparallel instead.", ShimWarning)


'/Users/ivar'

In [11]:
import pymc as pm
import numpy as np
import pymc.progressbar as pbar

def _parents_to_random_posterior_sample(bottom_node, pos=None):
    """Walks through parents and sets them to pos sample."""
    for i, parent in enumerate(bottom_node.extended_parents):
        if not isinstance(parent, pm.Node): # Skip non-stochastic nodes
            continue

        if pos is None:
            # Set to random posterior position
            pos = np.random.randint(0, len(parent.trace()))

        assert len(parent.trace()) >= pos, "pos larger than posterior sample size"
        parent.value = parent.trace()[pos]

def _post_pred_generate(bottom_node, samples=500, data=None, append_data=True):
    """Generate posterior predictive data from a single observed node."""
    datasets = []
    ##############################
    # Sample and generate stats
    for sample in range(samples):
        _parents_to_random_posterior_sample(bottom_node)
        # Generate data from bottom node
        sampled_data = bottom_node.random()
        if append_data and data is not None:
            sampled_data.reset_index(inplace=True)  # Only modification of original Kabuki code
            sampled_data = sampled_data.join(data.reset_index(), lsuffix='_sampled')
        datasets.append(sampled_data)
    return datasets

def post_pred_gen(model, groupby=None, samples=500, append_data=False, progress_bar=True):
    results = {}

    # Progress bar
    if progress_bar:
        n_iter = len(model.get_observeds())
        bar = pbar.progress_bar(n_iter)
        bar_iter = 0
    else:
        print("Sampling...")

    if groupby is None:
        iter_data = ((name, model.data.loc[obs['node'].value.index]) for name, obs in model.iter_observeds())
    else:
        iter_data = model.data.groupby(groupby)

    for name, data in iter_data:
        node = model.get_data_nodes(data.index)

        if progress_bar:
            bar_iter += 1
            bar.update(bar_iter)

        if node is None or not hasattr(node, 'random'):
            continue # Skip

        ##############################
        # Sample and generate stats
        datasets = _post_pred_generate(node, samples=samples, data=data, append_data=append_data)
        results[name] = pd.concat(datasets, names=['sample'], keys=list(range(len(datasets))))

    if progress_bar:
        bar_iter += 1
        bar.update(bar_iter)

    return pd.concat(results, names=['node'])

In [12]:
####### re-define functions we need for convinience  #######

# https://github.com/hddm-devs/kabuki/blob/master/kabuki/analyze.py#L420
def _parents_to_random_posterior_sample(bottom_node, pos=None):
    """Walks through parents and sets them to pos sample."""
    import pymc as pm
    import numpy as np
    for i, parent in enumerate(bottom_node.extended_parents):
        if not isinstance(parent, pm.Node): # Skip non-stochastic nodes
            continue

        if pos is None:
            # Set to random posterior position
            pos = np.random.randint(0, len(parent.trace()))

        assert len(parent.trace()) >= pos, "pos larger than posterior sample size"
        parent.value = parent.trace()[pos]

# https://github.com/hddm-devs/kabuki/blob/master/kabuki/analyze.py#L271
def _post_pred_generate(bottom_node, samples=500, data=None, append_data=False):
    """Generate posterior predictive data from a single observed node."""
    datasets = []

    ##############################
    # Sample and generate stats
    for sample in range(samples):
        _parents_to_random_posterior_sample(bottom_node)
        # Generate data from bottom node
        sampled_data = bottom_node.random()
        if append_data and data is not None:
            sampled_data = sampled_data.join(data.reset_index(), lsuffix='_sampled')
        datasets.append(sampled_data)

    return datasets

# https://github.com/hddm-devs/kabuki/blob/master/kabuki/analyze.py#L287
def post_pred_gen(model, groupby=None, samples=500, append_data=False, progress_bar=True):
    """Run posterior predictive check on a model.
    :Arguments:
        model : kabuki.Hierarchical
            Kabuki model over which to compute the ppc on.
    :Optional:
        samples : int
            How many samples to generate for each node.

        groupby : list
            Alternative grouping of the data. If not supplied, uses splitting
            of the model (as provided by depends_on).
        append_data : bool (default=False)
            Whether to append the observed data of each node to the replicatons.
        progress_bar : bool (default=True)
            Display progress bar
    :Returns:
        Hierarchical pandas.DataFrame with multiple sampled RT data sets.
        1st level: wfpt node
        2nd level: posterior predictive sample
        3rd level: original data index
    :See also:
        post_pred_stats
    """
    import pymc.progressbar as pbar
    results = {}

    # Progress bar
    if progress_bar:
        n_iter = len(model.get_observeds())
        bar = pbar.progress_bar(n_iter)
        bar_iter = 0
    else:
        print("Sampling...")

    if groupby is None:
        #### here I changed `iloc` to `loc`
        iter_data = ((name, model.data.loc[obs['node'].value.index]) for name, obs in model.iter_observeds())
    else:
        iter_data = model.data.groupby(groupby)


    for name, data in iter_data:
        node = model.get_data_nodes(data.index)

        if progress_bar:
            bar_iter += 1
            bar.update(bar_iter)

        if node is None or not hasattr(node, 'random'):
            continue # Skip

        ##############################
        # Sample and generate stats
        datasets = _post_pred_generate(node, samples=samples, data=data, append_data=append_data)
        results[name] = pd.concat(datasets, names=['sample'], keys=list(range(len(datasets))))

    if progress_bar:
        bar_iter += 1
        bar.update(bar_iter)


    return pd.concat(results, names=['node'])


In [2]:
cd='/Users/ivar/Documents/ddm_rules/'
os.chdir(cd)

In [4]:
model_1a = hddm.load("model_violate/model_violate")  # or whatever filename you saved with
model_1b = hddm.load("model_comply/model_comply")  # or whatever filename you saved with
model_1c = hddm.load("model_punishment/model_punishment")  # or whatever filename you saved with


In [14]:
# Generate posterior predictive samples
ppc_data_1a = post_pred_gen(model_1a, samples=1000, append_data=True)

 [-----------------100%-----------------] 479 of 476 complete in 11106.9 sec

In [16]:
# Generate posterior predictive samples
ppc_data_1b = post_pred_gen(model_1b, samples=1000, append_data=True)
# Generate posterior predictive samples
ppc_data_1c = post_pred_gen(model_1c, samples=1000, append_data=True)

 [-----------------100%-----------------] 487 of 484 complete in 12003.9 sec

In [34]:
# Observed: use the original rt and response columns
obs = ppc_data_1a[['rt', 'response', 'subj_idx', 'text', 'purpose']].copy()

# Simulated: rename sampled columns to match expected format
sim = ppc_data_1a[['rt_sampled', 'response_sampled', 'subj_idx', 'text', 'purpose']].copy()
sim = sim.rename(columns={'rt_sampled': 'rt', 'response_sampled': 'response'})

# stats_1a = hddm.utils.post_pred_stats(obs, sim)

# assuming you already have:
# vdata  -> observed data
# ppc_data -> simulated data
# model_name -> string, e.g. 'mymodel'

for text_level in obs['text'].unique():
    for purpose_level in obs['purpose'].unique():
        # Subset both observed and simulated data
        v_sub = obs[(obs['text'] == text_level) & (obs['purpose'] == purpose_level)]
        ppc_sub = ppc_data_1a[(ppc_data_1a['text'] == text_level) & (ppc_data_1a['purpose'] == purpose_level)]

        # Compute PPC stats for this subset
        ppc_stats = hddm.utils.post_pred_stats(v_sub, ppc_sub)

        # Save results with clear filename
        fname = f"study1a_ppc_stats_text-{text_level}_purpose-{purpose_level}.csv"
        ppc_stats.to_csv(fname, index=False)

        print(f"Saved: {fname}")

/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Saved: study1a_ppc_stats_text-0.0_purpose-0.0.csv


/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Saved: study1a_ppc_stats_text-0.0_purpose-1.0.csv


/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Saved: study1a_ppc_stats_text-1.0_purpose-0.0.csv


/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Saved: study1a_ppc_stats_text-1.0_purpose-1.0.csv


In [31]:
ppc_data_1a

rt_sampled  response_sampled  index  \
node                     sample                                           
wfpt(0.0.0.0).AATYL3173V 0      0     1.787959               1.0   4668   
                                1    -2.615259               0.0   4670   
                                2    -2.156259               0.0   4674   
                                3    -2.547859               0.0   4680   
                                4     2.298759               1.0   4685   
...                                        ...               ...    ...   
wfpt(1.0.1.0).ZXRVB5144P 999    16    2.738334               1.0   3506   
                                17    2.637434               1.0   3508   
                                18    2.969634               1.0   3514   
                                19    5.663134               1.0   3520   
                                20    2.876934               1.0   3521   

                                      study   subj_code strata_age  \
node                     sample                                      
wfpt(0.0.0.0).AATYL3173V 0      0   violate  AATYL3173V        NaN   
                                1   violate  AATYL3173V        NaN   
                                2   violate  AATYL3173V        NaN   
                                3   violate  AATYL3173V        NaN   
                                4   violate  AATYL3173V        NaN   
...                                     ...         ...        ...   
wfpt(1.0.1.0).ZXRVB5144P 999    16  violate  ZXRVB5144P        NaN   
                                17  violate  ZXRVB5144P        NaN   
                                18  violate  ZXRVB5144P        NaN   
                                19  violate  ZXRVB5144P        NaN   
                                20  violate  ZXRVB5144P        NaN   

                                    trial_index   task  scen_number  \
node                     sample                                       
wfpt(0.0.0.0).AATYL3173V 0      0            33  rules          NaN   
                                1            37  rules          NaN   
                                2            45  rules          NaN   
                                3            59  rules          NaN   
                                4            69  rules          NaN   
...                                         ...    ...          ...   
wfpt(1.0.1.0).ZXRVB5144P 999    16          197  rules          NaN   
                                17          201  rules          NaN   
                                18          215  rules          NaN   
                                19          227  rules          NaN   
                                20          229  rules          NaN   

                                    block_number  ... spd_acc_mode  \
node                     sample                   ...                
wfpt(0.0.0.0).AATYL3173V 0      0            NaN  ...          NaN   
                                1            NaN  ...          NaN   
                                2            NaN  ...          NaN   
                                3            NaN  ...          NaN   
                                4            NaN  ...          NaN   
...                                          ...  ...          ...   
wfpt(1.0.1.0).ZXRVB5144P 999    16           NaN  ...          NaN   
                                17           NaN  ...          NaN   
                                18           NaN  ...          NaN   
                                19           NaN  ...          NaN   
                                20           NaN  ...          NaN   

                                   spd_acc_order theory_question_binary  \
node                     sample                                           
wfpt(0.0.0.0).AATYL3173V 0      0            NaN                    NaN   
                                1            NaN                    NaN   
              

In [35]:
# Observed: use the original rt and response columns
obs = ppc_data_1b[['rt', 'response', 'subj_idx', 'text', 'purpose']].copy()

# Simulated: rename sampled columns to match expected format
sim = ppc_data_1b[['rt_sampled', 'response_sampled', 'subj_idx', 'text', 'purpose']].copy()
sim = sim.rename(columns={'rt_sampled': 'rt', 'response_sampled': 'response'})

# stats_1a = hddm.utils.post_pred_stats(obs, sim)

# assuming you already have:
# vdata  -> observed data
# ppc_data -> simulated data
# model_name -> string, e.g. 'mymodel'

for text_level in obs['text'].unique():
    for purpose_level in obs['purpose'].unique():
        # Subset both observed and simulated data
        v_sub = obs[(obs['text'] == text_level) & (obs['purpose'] == purpose_level)]
        ppc_sub = ppc_data_1b[(ppc_data_1b['text'] == text_level) & (ppc_data_1b['purpose'] == purpose_level)]

        # Compute PPC stats for this subset
        ppc_stats = hddm.utils.post_pred_stats(v_sub, ppc_sub)

        # Save results with clear filename
        fname = f"study1b_ppc_stats_text-{text_level}_purpose-{purpose_level}.csv"
        ppc_stats.to_csv(fname, index=False)

        print(f"Saved: {fname}")
        

# Observed: use the original rt and response columns
obs = ppc_data_1c[['rt', 'response', 'subj_idx', 'text', 'purpose']].copy()

# Simulated: rename sampled columns to match expected format
sim = ppc_data_1c[['rt_sampled', 'response_sampled', 'subj_idx', 'text', 'purpose']].copy()
sim = sim.rename(columns={'rt_sampled': 'rt', 'response_sampled': 'response'})

# stats_1a = hddm.utils.post_pred_stats(obs, sim)

# assuming you already have:
# vdata  -> observed data
# ppc_data -> simulated data
# model_name -> string, e.g. 'mymodel'

for text_level in obs['text'].unique():
    for purpose_level in obs['purpose'].unique():
        # Subset both observed and simulated data
        v_sub = obs[(obs['text'] == text_level) & (obs['purpose'] == purpose_level)]
        ppc_sub = ppc_data_1c[(ppc_data_1c['text'] == text_level) & (ppc_data_1c['purpose'] == purpose_level)]

        # Compute PPC stats for this subset
        ppc_stats = hddm.utils.post_pred_stats(v_sub, ppc_sub)

        # Save results with clear filename
        fname = f"study1c_ppc_stats_text-{text_level}_purpose-{purpose_level}.csv"
        ppc_stats.to_csv(fname, index=False)

        print(f"Saved: {fname}")

/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Saved: study1b_ppc_stats_text-0.0_purpose-0.0.csv


/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Saved: study1b_ppc_stats_text-0.0_purpose-1.0.csv


/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


...

Saved: study1c_ppc_stats_text-0.0_purpose-1.0.csv


/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Saved: study1c_ppc_stats_text-1.0_purpose-0.0.csv


/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/fromnumeric.py:3373: RuntimeWarning: Mean of empty slice.
  out=out, **kwargs)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:170: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:234: RuntimeWarning: Degrees of freedom <= 0 for slice
  keepdims=keepdims)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:195: RuntimeWarning: invalid value encountered in true_divide
  arrmean, rcount, out=arrmean, casting='unsafe', subok=False)
/Users/ivar/opt/anaconda3/envs/pyHDDM/lib/python3.6/site-packages/numpy/core/_methods.py:226: RuntimeWarning: invalid value encountered in double_scalars
  ret = ret.dtype.type(ret / rcount)


Saved: study1c_ppc_stats_text-1.0_purpose-1.0.csv
